### the intention here is to build the workflow where we gather data response from the llm about whats in the pdf guia, that was proven before that it can be accurate, in notebook 04, and then compare with the field DS_PROCEDIMENTO and see if they match or not, generating another df with a aut_validacao_status. we can t keep ds contato , so in the workflow we check if theres a analysis form the analist or not for comparing only purposes of the past data

# Part 1

load the dataframe into memory and bring it to a valid input for unstructured

### DS_CONTATO, meaning the data from query finalizado, we dont care in production stage anymore, so we dont use this base anymore

In [9]:
import sys

# add app to path so we dont get the error ModuleNotFoundError: No module named 'app'
from pathlib import Path
sys.path.append(str(Path.cwd().parent))
from app.services.database.oracle import OracleService
from app.services.database.mariadb import MariaDBService
from app.utils.db_operations import load_query_from_file, execute_query_to_df
from app.utils.config import load_config
from app.utils.logger import get_logger


logger = get_logger(name=__name__)
config_vars = load_config()

In [10]:
# load the query strings
autorizacao_query_file = "/home/joao/projects/company_projects/autorizacoes-api/POC/docs/querys/query_autorizacao.sql"
autorizacao_query_str = load_query_from_file(autorizacao_query_file)

procedimento_query_file = "/home/joao/projects/company_projects/autorizacoes-api/POC/docs/querys/query_procedimento.sql"
procedimento_query_str = load_query_from_file(procedimento_query_file)

finalizado_query_file = "/home/joao/projects/company_projects/autorizacoes-api/POC/docs/querys/query_pedido_finalizado_resposta.sql"
finalizado_query_str = load_query_from_file(finalizado_query_file)

aviso_cirurgia_query_file = "/home/joao/projects/company_projects/autorizacoes-api/POC/docs/querys/query_aviso_cirurgia.sql"
aviso_cirurgia_query_str = load_query_from_file(aviso_cirurgia_query_file)

In [12]:
maria_db_procedimento_df = execute_query_to_df(db_service_class=MariaDBService(settings=config_vars), query=procedimento_query_str, fetch_limit=500)
oracle_db_autorizacao_df = execute_query_to_df(db_service_class=OracleService(settings=config_vars), query=autorizacao_query_str)
oracle_db_finalizado_df = execute_query_to_df(db_service_class=OracleService(settings=config_vars), query=finalizado_query_str)
maria_db_aviso_cirurgia_df = execute_query_to_df(db_service_class=MariaDBService(settings=config_vars), query=aviso_cirurgia_query_str)

{"timestamp": "2025-08-27T17:27:49", "level": "INFO", "name": "app.services.database.mariadb", "message": "✅ MariaDB connection established to\n                srvawsdb002.cow7tj30bxpl.us-east-1.rds.amazonaws.com:3306/", "filename": "mariadb.py", "lineno": 27}
{"timestamp": "2025-08-27T17:27:49", "level": "INFO", "name": "app.services.database.mariadb", "message": "Executing MariaDB query...", "filename": "mariadb.py", "lineno": 67}
{"timestamp": "2025-08-27T17:27:49", "level": "INFO", "name": "app.services.database.mariadb", "message": "✅ Fetched 280 rows from MariaDB.", "filename": "mariadb.py", "lineno": 75}
{"timestamp": "2025-08-27T17:27:49", "level": "INFO", "name": "app.services.database.mariadb", "message": "✅ MariaDB connection closed.", "filename": "mariadb.py", "lineno": 41}
{"timestamp": "2025-08-27T17:27:49", "level": "INFO", "name": "app.utils.db_operations", "message": "✅ Query executed successfully with 280 sample records", "filename": "db_operations.py", "lineno": 60}


## getting a single table with the info we need

In [14]:
import pandas as pd

maria_db_procedimento_df_processed = maria_db_procedimento_df.rename(columns={"surgical_order_id": "CD_AVISO_CIRURGIA", "procedure": "DS_PROCEDIMENTO"})
maria_db_procedimento_df_processed.drop(columns=["hospitalization_type"], inplace=True)
maria_db_procedimento_df_processed = maria_db_procedimento_df_processed[maria_db_procedimento_df_processed["DS_PROCEDIMENTO"].notna()]
maria_db_procedimento_df_processed.reset_index(drop=True, inplace=True)

autorizacao_no_need_cols = [col for col in oracle_db_autorizacao_df.columns if col not in ["CD_AVISO_CIRURGIA", "CD_GUIA", "CD_SENHA", "DS_GUIA_PATH"]]
oracle_db_autorizacao_df_processed = oracle_db_autorizacao_df.drop(columns=autorizacao_no_need_cols)
oracle_db_autorizacao_df_processed = oracle_db_autorizacao_df_processed[oracle_db_autorizacao_df_processed["DS_GUIA_PATH"].notna()]
oracle_db_autorizacao_df_processed.reset_index(drop=True, inplace=True)

finalizado_no_need_cols = [col for col in oracle_db_finalizado_df.columns if col not in ["CD_REGISTRO_VINCULADO", "DS_CONTATO"]]
oracle_db_finalizado_df_processed = oracle_db_finalizado_df.drop(columns=finalizado_no_need_cols)
oracle_db_finalizado_df_processed.rename(columns={"CD_REGISTRO_VINCULADO": "CD_AVISO_CIRURGIA"}, inplace=True)
oracle_db_finalizado_df_processed = oracle_db_finalizado_df_processed[oracle_db_finalizado_df_processed["DS_CONTATO"].notna()]
oracle_db_finalizado_df_processed.reset_index(drop=True, inplace=True)

maria_db_aviso_cirurgia_df.rename(columns={"surgical_order_id": "CD_AVISO_CIRURGIA"}, inplace=True)
keep_cols = ["CD_AVISO_CIRURGIA", "health_insurance_name"]
maria_db_aviso_cirurgia_df = maria_db_aviso_cirurgia_df[keep_cols]
maria_db_aviso_cirurgia_df.reset_index(drop=True, inplace=True)

In [15]:
merged_df_autorizacao = pd.merge(
    maria_db_procedimento_df_processed,
    oracle_db_autorizacao_df_processed,
    on="CD_AVISO_CIRURGIA",
    how="inner"
)

merged_df_autorizacao_final = pd.merge(
    merged_df_autorizacao,
    oracle_db_finalizado_df_processed,
    on="CD_AVISO_CIRURGIA",
    how="inner"
)

final_extracted_df = pd.merge(
    merged_df_autorizacao_final,
    maria_db_aviso_cirurgia_df,
    on="CD_AVISO_CIRURGIA",
    how="inner"
)

In [17]:
final_extracted_df.head(3)

,CD_AVISO_CIRURGIA,DS_PROCEDIMENTO,CD_GUIA,CD_SENHA,DS_GUIA_PATH,DS_CONTATO,health_insurance_name
0,857416,"[{""code"":30101557,""description"":""EXCISAO E ROT...",19494856.0,None,https://cdns.overmind.ai/autorizacao-cemig-000...,\n Procedimento Autorizado\n Pacient...,CEMIG SAUDE
1,857416,"[{""code"":30101557,""description"":""EXCISAO E ROT...",19494856.0,None,https://cdns.overmind.ai/autorizacao-cemig-000...,\n Procedimento Autorizado\n Pacient...,CEMIG SAUDE
2,857469,"[{""code"":30907136,""description"":""VARIZES - TRA...",19495507.0,J5VEYT7,https://cdns.overmind.ai/autorizacao-bradesco-...,\n Procedimento Autorizado\n Pacient...,BRADESCO


In [18]:
# entering the pdf link, downloading the pdf, adding as another col
import requests
import numpy as np
def fetch_pdf_bytes(url):
    try:
        response = requests.get(url, timeout=15)
        if response.status_code == 200 and 'application/pdf' in response.headers.get('content-type', ''):
            return response.content
        else:
            return np.nan
    except Exception as e:
        print(f"Error fetching {url}: {e}")
        return np.nan

# Apply to all links in DS_GUIA_PATH
final_extracted_df['DS_PDF_BYTES'] = final_extracted_df['DS_GUIA_PATH'].apply(fetch_pdf_bytes)
final_extracted_df.head(3)

,CD_AVISO_CIRURGIA,DS_PROCEDIMENTO,CD_GUIA,CD_SENHA,DS_GUIA_PATH,DS_CONTATO,health_insurance_name,DS_PDF_BYTES
0,857416,"[{""code"":30101557,""description"":""EXCISAO E ROT...",19494856.0,None,https://cdns.overmind.ai/autorizacao-cemig-000...,\n Procedimento Autorizado\n Pacient...,CEMIG SAUDE,b'%PDF-1.3\n%\xb7\xbe\xad\xaa\n1 0 obj\n<<\n/T...
1,857416,"[{""code"":30101557,""description"":""EXCISAO E ROT...",19494856.0,None,https://cdns.overmind.ai/autorizacao-cemig-000...,\n Procedimento Autorizado\n Pacient...,CEMIG SAUDE,b'%PDF-1.3\n%\xb7\xbe\xad\xaa\n1 0 obj\n<<\n/T...
2,857469,"[{""code"":30907136,""description"":""VARIZES - TRA...",19495507.0,J5VEYT7,https://cdns.overmind.ai/autorizacao-bradesco-...,\n Procedimento Autorizado\n Pacient...,BRADESCO,b'%PDF-1.4\n%\xe2\xe3\xcf\xd3\n4 0 obj\n<</Len...
